# Paper 001: interactive AAD experiment workflow

Run the Python cells from top to bottom. The notebook uses the remote Colab
A100, a small code clone under /content, and the existing KaggleHub dataset
cache. It does not download AAD.

## 1. Confirm the A100 runtime

In [ ]:
import subprocess

import torch

subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
)
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access CUDA")
gpu_name = torch.cuda.get_device_name(0)
if "A100" not in gpu_name:
    raise RuntimeError(f"Expected an A100 runtime, found: {gpu_name}")
print("PyTorch CUDA device:", gpu_name)

## 2. Prepare code and locate the existing dataset

The notebook is local, but its kernel and filesystem are remote. The repository
is therefore cloned once into /content; later runs only pull updates.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/sanelehlabisa/phd_research.git"
REPOSITORY_ROOT = Path("/content/phd_research")
IMPLEMENTATION_DIR = (
    REPOSITORY_ROOT
    / "papers"
    / "001-journal-abnormal-activity-recognition"
    / "implementation"
)

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=REPOSITORY_ROOT,
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
    )

required_packages = {"av": "av==16.1.0", "torchmetrics": "torchmetrics==1.8.2"}
missing_packages = [
    package
    for module, package in required_packages.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )

versions_dir = Path(
    "/root/.cache/kagglehub/datasets/sanelehlabisa/"
    "abnormal-activities-dataset/versions"
)
dataset_candidates = [
    path
    for path in versions_dir.glob("*/abnormal-activities-dataset")
    if path.is_dir()
]
if not dataset_candidates:
    raise FileNotFoundError(f"Existing cached AAD dataset not found under {versions_dir}")
DATASET_DIR = max(
    dataset_candidates,
    key=lambda path: int(path.parent.name) if path.parent.name.isdigit() else -1,
).resolve()
RUNS_DIR = IMPLEMENTATION_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(IMPLEMENTATION_DIR)
if str(IMPLEMENTATION_DIR) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_DIR))

print("Implementation:", IMPLEMENTATION_DIR)
print("Dataset:", DATASET_DIR)
print("Runs:", RUNS_DIR)
print("Installed missing packages:", missing_packages or "none")

## 3. Inspect one clean and augmented video

This loads one real clip and displays its first, middle, and final frames.

In [ ]:
import matplotlib.pyplot as plt

from src.dataset import AHARDataset, VideoAugmentation

dataset = AHARDataset(
    DATASET_DIR,
    sequence_length=16,
    frame_size=(32, 32),
)
if not dataset.samples:
    raise ValueError("AAD contains no supported video samples")

sample_index = 0
sample_video, sample_label = dataset[sample_index]
augmented_video = VideoAugmentation()(sample_video.clone())
frame_indices = [0, len(sample_video) // 2, len(sample_video) - 1]

figure, axes = plt.subplots(2, 3, figsize=(10, 6))
for column, frame_index in enumerate(frame_indices):
    axes[0, column].imshow(sample_video[frame_index].permute(1, 2, 0).cpu())
    axes[0, column].set_title(f"Clean frame {frame_index}")
    axes[1, column].imshow(augmented_video[frame_index].permute(1, 2, 0).cpu())
    axes[1, column].set_title(f"Augmented frame {frame_index}")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
figure.suptitle(f"Class: {dataset.class_names[sample_label]}")
figure.tight_layout()
plt.show()

print("Samples:", len(dataset))
print("Classes:", dataset.class_names)
print("Video shape:", tuple(sample_video.shape))
print("Value range:", float(sample_video.min()), float(sample_video.max()))

## 4. Inspect the model and architecture candidates

In [ ]:
import json

import pandas as pd
from IPython.display import display

from src.model import CustomConvLSTM, count_trainable_parameters

device = torch.device("cuda")
model = CustomConvLSTM(
    dataset.num_classes,
    layers=[(8, (3, 3)), (16, (3, 3))],
).to(device).eval()
with torch.inference_mode():
    logits = model(sample_video.unsqueeze(0).to(device))

print(model)
print("Device:", torch.cuda.get_device_name(0))
print("Trainable parameters:", count_trainable_parameters(model))
print("Output shape:", tuple(logits.shape))

manifest_path = IMPLEMENTATION_DIR / "configs/aad_architecture_candidates.json"
manifest = json.loads(manifest_path.read_text())
candidate_rows = [
    {
        "name": candidate["name"],
        "filters": "-".join(
            str(layer[0]) for layer in candidate["convlstm_layers"]
        ),
        "research question": candidate["research_question"],
    }
    for candidate in manifest["candidates"]
]
display(pd.DataFrame(candidate_rows))

del model, logits
torch.cuda.empty_cache()

## 5. Run one controlled experiment stage

Start with architecture-screen. Change the stage later for baseline-confirmation,
published-topology, or focused-ablations.

In [ ]:
import shlex

EXPERIMENT_STAGE = "architecture-screen"
allowed_stages = {
    "architecture-screen",
    "baseline-confirmation",
    "published-topology",
    "focused-ablations",
}
if EXPERIMENT_STAGE not in allowed_stages:
    raise ValueError(f"Unknown experiment stage: {EXPERIMENT_STAGE}")

command = [
    sys.executable,
    "-u",
    "-m",
    "src.experiments",
    "--plan-config",
    str(IMPLEMENTATION_DIR / "configs/aad_controlled_experiment_plan.json"),
    "--run-plan-stage",
    EXPERIMENT_STAGE,
    "--plan-dataset-dir",
    str(DATASET_DIR),
    "--plan-runs-dir",
    str(RUNS_DIR),
]
print("$", shlex.join(command), flush=True)
completed = subprocess.run(command, cwd=IMPLEMENTATION_DIR, check=False)
if completed.returncode != 0:
    raise RuntimeError(f"Experiment command failed with exit code {completed.returncode}")

## 6. Display the latest completed experiment ranking

In [ ]:
summary_paths = list((RUNS_DIR / "experiments").glob("*/summary.json"))
if not summary_paths:
    raise FileNotFoundError("No completed experiment summary was found")
latest_summary_path = max(summary_paths, key=lambda path: path.stat().st_mtime)
latest_summary = json.loads(latest_summary_path.read_text())
ranking_rows = []
for rank, result in enumerate(latest_summary["ranked"], start=1):
    validation = result["validation_metrics"]
    ranking_rows.append(
        {
            "rank": rank,
            "model": result["name"],
            "macro_f1": validation["macro_f1"],
            "accuracy": validation["accuracy"],
            "parameters": result["num_params"],
            "train_time_s": result["train_time_s"],
        }
    )
print("Summary:", latest_summary_path)
display(pd.DataFrame(ranking_rows))

Artifacts remain under implementation/runs/ on the Colab machine. Architecture
selection uses validation results only; final test evaluation is not included.